# **Converting ERA5 .nc data to .csv with Essential Variables**
**Note:** ERA5 weather data was quite large, with the raw download totaling 14.4 MB. Only the processed CSV files were saved.

**Outputs** (saved under ``Open_datasets``)
- `data_stream-oper_stepType_combined.csv`
- `era5_melbourne.csv`

**Start Date:** *Feb 28, 2025*<br>
**Last Update:** *Mar 9, 2025*  
**Authors:** *Kanaha Shoji*

In [1]:
import xarray as xr
import glob
import os
import pandas as pd
from datetime import datetime
import numpy as np

/Users/shoji/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/shoji/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
download_dir = '/Users/shoji/Downloads'
# Set directory to where open data are stored
open_data_dir = os.path.join('/Users/shoji/Library/CloudStorage/OneDrive-epfl.ch/2025_03_Melbourne_walkability_study_final/Open_datasets')

Data downloaded from: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview

Our request for data:

- **Product type**: Reanalysis
- **Variable**: 10m u-component of wind, 10m v-component of wind, 2m dewpoint temperature, 2m temperature, Total precipitation, Total cloud cover
- **Year**: 2009 to 2019 (only 2 years each can be downloaded because of file size limit)
- **Month**: All selected
- **Day**: All selected
- **Time**: All selected
- **Geographical area**: (based on sensor locations. How we obtained them can be found in `data_processing.ipynb`)
  - North: -37.79698741°
  - West: 144.93970694°
  - South: -37.82401776°
  - East: 144.97323591°
- **Data format**: NetCDF4 (Experimental)
- **Download format**: Unarchived (not zipped if single file)

With this, each data downloaded was 2.65 MB. Each download consisted of:
- `data_stream-oper_stepType-accum.nc` (1.2 MB each for 2 years) contains Total precipitation
- `data_stream-oper_stepType-instant.nc` (1.4 MB each for 2 years) contains the rest of the variables

In [3]:
# Get all .nc files in any subdirectory under download_dir
nc_tp_files = glob.glob(os.path.join(download_dir, "**", "data_stream-oper_stepType-accum.nc"), recursive=True)

# Load all NetCDF files
ds_tp_list = [xr.open_dataset(f) for f in nc_tp_files]

# Merge all files along the time dimension (valid_time)
merged_ds_tp = xr.concat(ds_tp_list, dim="valid_time")

# Convert to DataFrame
df_tp_raw = merged_ds_tp.to_dataframe().reset_index()
df_tp_raw.drop(columns=['latitude', 'longitude','number','expver'], inplace=True)
print(df_tp_raw.head())

/Users/shoji/opt/anaconda3/lib/python3.9/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


           valid_time   tp
0 2019-01-01 00:00:00  0.0
1 2019-01-01 01:00:00  0.0
2 2019-01-01 02:00:00  0.0
3 2019-01-01 03:00:00  0.0
4 2019-01-01 04:00:00  0.0


In [4]:
# Get all .nc files in any subdirectory under download_dir
nc_files = glob.glob(os.path.join(download_dir, "**", "data_stream-oper_stepType-instant.nc"), recursive=True)

# Load all NetCDF files
ds_list = [xr.open_dataset(f) for f in nc_files]

# Merge all files along the time dimension (valid_time)
merged_ds = xr.concat(ds_list, dim="valid_time")

# Convert to DataFrame
df_raw = merged_ds.to_dataframe().reset_index()
df_raw.drop(columns=['latitude', 'longitude','number','expver'], inplace=True)
print(df_raw.head())

           valid_time       u10       v10         d2m         t2m  tcc
0 2019-01-01 00:00:00  0.213027  2.674958  287.087891  294.392578  0.0
1 2019-01-01 01:00:00  0.127945  2.711418  287.279785  295.179443  0.0
2 2019-01-01 02:00:00  0.133733  2.840489  287.474854  296.164795  0.0
3 2019-01-01 03:00:00  0.106432  3.266558  287.066895  297.439453  0.0
4 2019-01-01 04:00:00  0.066003  4.178085  285.916748  297.741211  0.0


In [5]:
# Put the two dataframes together
df_tp_raw['valid_time'] = pd.to_datetime(df_tp_raw['valid_time'])
df_raw['valid_time'] = pd.to_datetime(df_raw['valid_time'])
merged_raw_df = pd.merge(df_tp_raw, df_raw, on='valid_time', how='inner')
print(merged_raw_df.head())

           valid_time   tp       u10       v10         d2m         t2m  tcc
0 2019-01-01 00:00:00  0.0  0.213027  2.674958  287.087891  294.392578  0.0
1 2019-01-01 01:00:00  0.0  0.127945  2.711418  287.279785  295.179443  0.0
2 2019-01-01 02:00:00  0.0  0.133733  2.840489  287.474854  296.164795  0.0
3 2019-01-01 03:00:00  0.0  0.106432  3.266558  287.066895  297.439453  0.0
4 2019-01-01 04:00:00  0.0  0.066003  4.178085  285.916748  297.741211  0.0


In [6]:
merged_raw_df.to_csv(os.path.join(open_data_dir, 'data_stream-oper_stepType_combined.csv'), index=False)
merged_raw_df = pd.read_csv(os.path.join(open_data_dir, 'data_stream-oper_stepType_combined.csv'))

In [7]:
era5_df = merged_raw_df.copy()
# Convert temperature to Celsius from Kelvin
era5_df['temperature'] = era5_df['t2m'] - 273.15

In [9]:
# Calculate wind speed from u10 and v10 components
era5_df['ws'] = np.sqrt(era5_df['u10']**2 + era5_df['v10']**2)

# Convert precipitation unit from m to mm
era5_df['total_precipitation'] = (era5_df['tp']*1000).round(3) #convert from m to mm

# Round numbers and rename other weather-related columns for clarity
era5_df['temperature'] = era5_df['temperature'].round(1) # unit: °C
era5_df['wind_speed'] = era5_df['ws'].round(2) # unit: m/s
era5_df['cloud_cover'] = era5_df['tcc'].round(2)

# Add columns for merging later
era5_df['time'] = era5_df['valid_time'].apply(lambda x: datetime.strptime(x,'%Y-%m-%d %H:%M:%S'))
era5_df['Usable Year'] = era5_df['time'].apply(lambda x: x.year)
era5_df['month'] = era5_df['time'].apply(lambda x: x.month)
era5_df['day'] = era5_df['time'].apply(lambda x: x.day)
era5_df['hour'] = era5_df['time'].apply(lambda x: x.hour)


# Drop unnecessary columns
era5_df.drop(columns=['valid_time', 't2m', 'd2m', 'u10', 'v10', 'tp', 'ws','tcc'], inplace=True)

In [10]:
print(era5_df.head())

   temperature  total_precipitation  wind_speed  cloud_cover  \
0         21.2                  0.0        2.68          0.0   
1         22.0                  0.0        2.71          0.0   
2         23.0                  0.0        2.84          0.0   
3         24.3                  0.0        3.27          0.0   
4         24.6                  0.0        4.18          0.0   

                 time  Usable Year  month  day  hour  
0 2019-01-01 00:00:00         2019      1    1     0  
1 2019-01-01 01:00:00         2019      1    1     1  
2 2019-01-01 02:00:00         2019      1    1     2  
3 2019-01-01 03:00:00         2019      1    1     3  
4 2019-01-01 04:00:00         2019      1    1     4  


In [11]:
era5_df.to_csv(os.path.join(open_data_dir, 'era5_melbourne.csv'), index=False)